In [1]:
#!/usr/bin/env python3
"""
Run Backtest + Analyze - InfluxDB version
"""

from datetime import datetime
from ostrad_engine import (
    OSTRADBacktester, InfluxDBDataLoader,
    load_params, create_symbol_params, setup_logging
)
import pandas as pd

# ── Configuration ─────────────────────────────────────────────────────────────
INFLUX_URL   = "http://165.22.215.65:8086"
INFLUX_TOKEN = "1cRqCHl8Zvn-l8Hc1uznO-YTZ5I8PMIOCVVTcSrnuExaQHR0ZfsfKav_ubajrM2R3gEmH7o5v93OqWG-_ANpQw=="
INFLUX_ORG   = "BBC India"
BUCKET       = "test_nifty_options"   # both spot (fut_spot_merged) and options (options_1min) live here

SYMBOL     = "SENSEX"                 # change to NIFTY, BANKNIFTY etc.
START_DATE = datetime(2026, 4, 15)
END_DATE   = datetime(2026, 4, 15)
# ──────────────────────────────────────────────────────────────────────────────


def analyze_trades(trades_df: pd.DataFrame):
    print("\n" + "=" * 100)
    print("TRADE ANALYSIS")
    print("=" * 100)

    print(f"\nTotal Trades: {len(trades_df)}")
    print(f"Hedges: {trades_df['is_hedge'].sum()}")
    print(f"Shorts: {len(trades_df) - trades_df['is_hedge'].sum()}")

    for is_hedge in [False, True]:
        type_name = "HEDGE (BUY)" if is_hedge else "SHORT (SELL)"
        type_df = trades_df[trades_df['is_hedge'] == is_hedge]
        if type_df.empty:
            continue

        print(f"\n{'='*100}")
        print(f"{type_name} POSITIONS")
        print(f"{'='*100}")

        for _, trade in type_df.iterrows():
            print(f"\nStrike: {trade['strike']} {trade['option_type']}")
            print(f"  Entry: {trade['entry_time']} @ Rs.{trade['entry_price']:.2f}")
            print(f"  Qty: {trade['quantity']} ({trade['lots']} lots)")
            if not is_hedge:
                if pd.notna(trade['sl_price']):
                    print(f"  SL1: Rs.{trade['sl_price']:.2f}")
                if pd.notna(trade['sl2_price']):
                    print(f"  SL2: Rs.{trade['sl2_price']:.2f}")
            print(f"  Exit: {trade['exit_time']} @ Rs.{trade['exit_price']:.2f}")
            print(f"  Reason: {trade['exit_reason']}")
            print(f"  P&L: Rs.{trade['pnl']:,.2f}")

    print(f"\n{'='*100}")
    print("EXIT REASON BREAKDOWN")
    print(f"{'='*100}")
    summary = trades_df.groupby('exit_reason')['pnl'].agg(['count', 'sum', 'mean']).round(2)
    print(summary)

    print(f"\n{'='*100}")
    print(f"TOTAL P&L: Rs.{trades_df['pnl'].sum():,.2f}")
    print(f"{'='*100}")


def main():
    logger = setup_logging(f'{SYMBOL.lower()}_influx_backtest.log')
    all_params = load_params('params.json')
    params = create_symbol_params(all_params['global'], all_params['symbols'][SYMBOL])

    print("=" * 60)
    print("STEP 1: LOADING DATA FROM INFLUXDB")
    print("=" * 60)

    loader = InfluxDBDataLoader(
        url=INFLUX_URL,
        token=INFLUX_TOKEN,
        org=INFLUX_ORG,
        spot_bucket=BUCKET,
        option_bucket=BUCKET,
        symbol=SYMBOL,
        preload_start=START_DATE,
        preload_end=END_DATE,
        logger=logger,
    )

    if loader.spot_df is None or loader.spot_df.empty:
        print("ERROR: No spot data returned from InfluxDB.")
        loader.close()
        return

    if loader.options_df is None or loader.options_df.empty:
        print("ERROR: No options data returned from InfluxDB.")
        loader.close()
        return

    print(f"Spot   : {loader.spot_df.shape[0]} rows  {loader.spot_df.index.min()} → {loader.spot_df.index.max()}")
    print(f"Options: {loader.options_df.shape[0]} rows  {loader.options_df.index.min()} → {loader.options_df.index.max()}")

    print("\n" + "=" * 60)
    print("STEP 2: RUNNING BACKTEST")
    print("=" * 60)

    backtester = OSTRADBacktester(SYMBOL, params, loader, logger)
    results = backtester.run_backtest(START_DATE, END_DATE)

    files = backtester.save_results(results, f'results/{SYMBOL.lower()}_influx')

    print("\n" + "=" * 60)
    print("STEP 3: ANALYZING TRADES")
    print("=" * 60)

    trades_df = pd.DataFrame(results['trades'])
    if trades_df.empty:
        print("No trades to analyze.")
    else:
        analyze_trades(trades_df)

    loader.close()
    return results


results = main()


STEP 1: LOADING DATA FROM INFLUXDB


2026-04-22 14:47:18,519 | INFO | Loaded 375 spot records, 197797 option records from InfluxDB


Spot   : 375 rows  2026-04-15 03:45:00 → 2026-04-15 09:59:00
Options: 197797 rows  2026-04-15 03:45:00 → 2026-04-15 09:59:00

STEP 2: RUNNING BACKTEST


TypeError: MarginConfig.__init__() got an unexpected keyword argument 'var_margin_percent'. Did you mean 'margin_percent'?